# 05 - Evaluacion sobre Test (held-out independiente)

- **M1/M2**: carpeta `splits/test/` con **doble entrada** (original + hoja aislada), igual que en entrenamiento e inferencia.
- **M_seg**: **recall de hoja** (principal) + Dice/IoU vs mascaras expertas COCO (`splits/masks/`, test 25% seed 42). La red solo decide hoja/fondo.

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn opencv-python-headless pycocotools

In [ ]:
import json
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, recall_score, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

OUT = Path('./outputs'); SPLIT = Path('./splits')
m1 = tf.keras.models.load_model(OUT / 'model1_binary.keras', compile=False)
m2 = tf.keras.models.load_model(OUT / 'model2_pathogen.keras', compile=False)
print('M1/M2 cargados (dual-input)')

In [ ]:
import time
import numpy as np
import cv2
from PIL import Image
from pathlib import Path
from tensorflow.keras.applications.efficientnet import preprocess_input

_HSV_GREEN_LOW  = np.array([20, 25, 30]); _HSV_GREEN_HIGH = np.array([90, 255, 255])
_HSV_LESION_LOW = np.array([0, 40, 25]);  _HSV_LESION_HIGH = np.array([35, 255, 230])
_MORPH = np.ones((7, 7), np.uint8)


def pseudo_leaf_mask(img_rgb):
    hsv = cv2.cvtColor(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2HSV)
    leaf = cv2.bitwise_or(cv2.inRange(hsv, _HSV_GREEN_LOW, _HSV_GREEN_HIGH),
                          cv2.inRange(hsv, _HSV_LESION_LOW, _HSV_LESION_HIGH))
    leaf = cv2.morphologyEx(leaf, cv2.MORPH_CLOSE, _MORPH)
    return cv2.morphologyEx(leaf, cv2.MORPH_OPEN, _MORPH)


def leaf_isolate(img_rgb):
    m = pseudo_leaf_mask(img_rgb); out = img_rgb.copy(); out[m == 0] = 0; return out


def chromatic_normalize(img_rgb):
    x = img_rgb.astype(np.float32); il = np.power(np.mean(np.power(x, 6), axis=(0, 1)), 1.0 / 6.0)
    return np.clip(x * np.clip(il.mean() / (il + 1e-6), 0.6, 1.6), 0, 255).astype(np.uint8)


def _list(directory):
    classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
    idx = {c: i for i, c in enumerate(classes)}
    items = []
    for c in classes:
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            for fp in (Path(directory) / c).glob(ext):
                items.append((str(fp), idx[c]))
    return idx, items


def predict_dual(model, directory, size, batch=32):
    idx, items = _list(directory)
    probs, ytrue = [], []
    ob, lb, yb = [], [], []
    def flush():
        if not ob: return
        p = model.predict([np.stack(ob), np.stack(lb)], verbose=0)
        probs.append(p); ytrue.extend(yb)
    for fp, y in items:
        img = np.array(Image.open(fp).convert('RGB').resize(size))
        ob.append(preprocess_input(img.astype(np.float32)))
        lb.append(preprocess_input(leaf_isolate(img).astype(np.float32)))
        yb.append(y)
        if len(ob) == batch:
            flush(); ob, lb, yb = [], [], []
    flush()
    return idx, np.array(ytrue), np.concatenate(probs) if probs else np.zeros((0, 1))

In [ ]:
idx1, y1, p1 = predict_dual(m1, SPLIT / 'test' / 'clasificacion_binaria', (240, 240))
enf_idx = next((i for c, i in idx1.items() if 'enferm' in c.lower()), 0)
names1 = [c for c, _ in sorted(idx1.items(), key=lambda kv: kv[1])]
p_pos1 = p1.reshape(-1)
p_enf = p_pos1 if enf_idx == 1 else 1.0 - p_pos1
y_enf = (y1 == enf_idx).astype(int)
pred_enf = (p_enf >= 0.5).astype(int)
pred_idx = np.where(pred_enf == 1, enf_idx, 1 - enf_idx)

acc1 = accuracy_score(y1, pred_idx)
rec1 = recall_score(y_enf, pred_enf, zero_division=0)
f1e = f1_score(y_enf, pred_enf, zero_division=0)
print('=== Modelo 1 (dual-input) ===')
print(classification_report(y1, pred_idx, target_names=names1, digits=4, zero_division=0))
print(f'accuracy={acc1:.4f} | recall enferma={rec1:.4f} | F1 enferma={f1e:.4f}')
cm1 = confusion_matrix(y1, pred_idx)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', xticklabels=names1, yticklabels=names1)
plt.title('M1 - confusion'); plt.ylabel('real'); plt.xlabel('pred'); plt.tight_layout(); plt.savefig(OUT / 'cm_m1.png', dpi=120); plt.show()

In [ ]:
idx2, y2, probs2 = predict_dual(m2, SPLIT / 'test' / 'clasificacion_patogeno', (224, 224))
names2 = [c for c, _ in sorted(idx2.items(), key=lambda kv: kv[1])]
yp2 = np.argmax(probs2, 1)
acc2 = float(accuracy_score(y2, yp2))
f1m = float(f1_score(y2, yp2, average='macro', zero_division=0))
per = f1_score(y2, yp2, average=None, labels=range(len(names2)), zero_division=0)
per_class_f1 = {n: float(per[i]) for i, n in enumerate(names2)}
print('=== Modelo 2 (dual-input single-label) ===')
print(f'accuracy={acc2:.4f} | F1 macro={f1m:.4f}')
print(classification_report(y2, yp2, labels=range(len(names2)), target_names=names2, digits=4, zero_division=0))
cm2 = confusion_matrix(y2, yp2, labels=range(len(names2)))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', xticklabels=names2, yticklabels=names2)
plt.title('M2 - confusion'); plt.ylabel('real'); plt.xlabel('pred'); plt.tight_layout(); plt.savefig(OUT / 'cm_m2.png', dpi=120); plt.show()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(names2)), [per_class_f1[n] for n in names2], color='steelblue')
ax.set_xticks(range(len(names2))); ax.set_xticklabels(names2, rotation=30); ax.set_ylim(0, 1); ax.set_title('M2 - F1 por clase'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / 'm2_per_class_metrics.png', dpi=120); plt.show()

In [ ]:
metrics = {
    'm1': {'accuracy': float(acc1), 'recall_enferma': float(rec1), 'f1_enferma': float(f1e)},
    'm2': {'accuracy': acc2, 'f1_macro': f1m, 'per_class_f1': per_class_f1},
}
json.dump(metrics, open(OUT / 'training_metrics.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print('M1 acc>=0.98', acc1 >= 0.98, '| recall>=0.97', rec1 >= 0.97)
print('M2 acc>=0.9514', acc2 >= 0.9514, '| F1 macro>=0.9472', f1m >= 0.9472)

In [ ]:
from pycocotools.coco import COCO

MASKS_DIR = SPLIT / 'masks'; COCO_ANN = MASKS_DIR / '_annotations.coco.json'
mseg = tf.keras.models.load_model(OUT / 'model_seg.keras', compile=False)


def _coco_test_ids(coco, frac=0.25, seed=42):
    ids = sorted(coco.getImgIds()); np.random.RandomState(seed).shuffle(ids)
    return ids[:round(len(ids) * frac)]


def _coco_leaf(coco, img_id, size):
    info = coco.loadImgs(img_id)[0]
    leaf = np.zeros((info['height'], info['width']), np.uint8)
    for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
        leaf = np.maximum(leaf, coco.annToMask(ann))
    return info, cv2.resize(leaf, size, interpolation=cv2.INTER_NEAREST)


if COCO_ANN.exists():
    coco = COCO(str(COCO_ANN)); test_ids = _coco_test_ids(coco)
    recs, dices, ious = [], [], []
    for img_id in test_ids:
        info, gt = _coco_leaf(coco, img_id, (256, 256))
        raw = np.array(Image.open(MASKS_DIR / info['file_name']).convert('RGB').resize((256, 256)))
        norm = chromatic_normalize(raw)
        pred = (np.argmax(mseg.predict((norm.astype(np.float32) / 255.0)[np.newaxis], verbose=0)[0], -1) == 1).astype(np.uint8)
        g = (gt == 1).astype(np.uint8)
        inter = int(np.sum((pred == 1) & (g == 1))); ps, gs = int(pred.sum()), int(g.sum())
        recs.append((inter + 1e-6) / (gs + 1e-6)); dices.append((2 * inter + 1e-6) / (ps + gs + 1e-6)); ious.append((inter + 1e-6) / (ps + gs - inter + 1e-6))
    print(f'=== M_seg vs experto COCO (n={len(test_ids)}, held-out 25% seed=42) ===')
    print(f'Recall hoja (principal): {np.mean(recs):.3f}')
    print(f'Dice hoja  (secundario): {np.mean(dices):.3f}')
    print(f'IoU  hoja  (secundario): {np.mean(ious):.3f}')
else:
    print('WARNING: no se encontro', COCO_ANN)